# TP Integrador — Modulación LoRa / FSCM

Grupo 8 — Testigos de Vangelista. El notebook está ordenado por bloques, con una
celda por subpunto del transmisor y del receptor.


In [ ]:
# TRANSMISOR — 1. Parámetros del PHY y del preámbulo
import numpy as np
import matplotlib.pyplot as plt

SF = 8                          # bits por símbolo
SamplingRate = 1_000_000        # tasa del SDR [muestras/s]
B = SamplingRate                # el chirp barre exactamente la tasa de muestreo
M = 2 ** SF                     # muestras por símbolo = símbolos posibles
T = 1 / B                       # período de chip [s]
Ts = M * T                      # duración de un símbolo [s]
k = np.arange(M)                # índice de chip

fc = 915e6                      # portadora dentro de la banda 900-920 MHz

n_up = 8                        # up-chirps del preámbulo
n_down = 2                      # down-chirps completos
cuarto_down = True              # último 1/4 de down-chirp
n_simbolos = 10                 # símbolos del payload

tramas_en_buffer = 2            # tramas por captura del receptor
n_tramas_fer = 30000            # tramas totales por curva de FER
snr_fer_db = np.arange(-12, 0, 1)   # SNR = Eb/N0 [dB], de -12 a -1
semilla = 2024

print(f"SF = {SF} | M = {M} | B = {B/1e6:.3f} MHz | fc = {fc/1e6:.0f} MHz")
print(f"Símbolo = {Ts*1e6:.1f} µs | Preámbulo = {(n_up + n_down)*M + (M//4 if cuarto_down else 0)} muestras")
print(f"FER: {n_tramas_fer} tramas en total sobre {snr_fer_db.size} puntos de SNR")

In [ ]:
# TRANSMISOR — 2. De bits a símbolo (Ecuación 1)
def bits_a_simbolo(bits, SF=SF):
    """Convierte filas de bits (LSB primero) en símbolos enteros."""
    simbolos = []
    for fila in np.asarray(bits):
        fila = np.asarray(fila, dtype=np.int64).ravel()
        if fila.size != SF:
            raise ValueError("cada fila debe tener SF bits")
        simbolos.append(int(np.dot(fila, 2 ** np.arange(SF))))
    return np.asarray(simbolos)


generador_bits = np.random.default_rng(semilla)
bits_payload = generador_bits.integers(0, 2, size=(n_simbolos, SF))
simbolos_payload = bits_a_simbolo(bits_payload)

print("Bits del payload (LSB primero) -> símbolo:")
for fila, simbolo in zip(bits_payload, simbolos_payload):
    print(f"  {fila.tolist()} -> {simbolo:3d}")

In [ ]:
# TRANSMISOR — 3. Modulación de símbolo (Ecuaciones 2 y 3)
def up_chirp(M=M):
    """Chirp ascendente base, energía unitaria."""
    kk = np.arange(M)
    return (1.0 / np.sqrt(M)) * np.exp(1j * 2 * np.pi * (kk ** 2) / M)


def down_chirp(M=M):
    """Chirp descendente: el conjugado del ascendente."""
    return np.conj(up_chirp(M))


def down_chirp_demod(M=M):
    """Réplica usada para dechirpear (sin la escala 1/sqrt(M))."""
    kk = np.arange(M)
    return np.exp(-1j * 2 * np.pi * (kk ** 2) / M)


def modular(simbolos, M=M):
    """Genera la señal FSCM de una secuencia de símbolos."""
    kk = np.arange(M)
    partes = []
    for simbolo in np.asarray(simbolos).ravel():
        if not 0 <= int(simbolo) < M:
            raise ValueError(f"símbolo fuera de rango: {simbolo}")
        fase = 2 * np.pi * (((int(simbolo) + kk) % M) * kk) / M
        partes.append((1 / np.sqrt(M)) * np.exp(1j * fase))
    return np.concatenate(partes)


def preambulo(M=M, n_up=n_up, n_down=n_down, cuarto=cuarto_down):
    """Preámbulo: up-chirps, down-chirps y 1/4 de down-chirp."""
    partes = [np.tile(up_chirp(M), n_up)]
    if n_down:
        partes.append(np.tile(down_chirp(M), n_down))
    if cuarto:
        partes.append(down_chirp(M)[: M // 4])
    return np.concatenate(partes)


def longitud_preambulo(M=M, n_up=n_up, n_down=n_down, cuarto=cuarto_down):
    largo = n_up * M + n_down * M
    if cuarto:
        largo += M // 4
    return largo

In [ ]:
# TRANSMISOR — 4. Gráfico de la señal del preámbulo
def plotSignal(signal):
    #------------------------------- Componente en fase -------------------------------
    plt.figure(figsize=(20, 8), dpi=80, facecolor="w", edgecolor="k")
    plt.subplot(2, 1, 1)
    plt.plot(np.real(signal))
    plt.xlabel("n")
    plt.ylabel("Amplitud")
    plt.title("Componente en fase de la señal")
    plt.grid()
    #------------------------------- Componente en cuadratura ------------------------
    plt.subplot(2, 1, 2)
    plt.plot(np.imag(signal))
    plt.xlabel("n")
    plt.ylabel("Amplitud")
    plt.title("Componente en cuadratura de la señal")
    plt.grid()
    plt.show()


preambulo_ref = preambulo()
plotSignal(preambulo_ref)

In [ ]:
# TRANSMISOR — 5. Densidad espectral de potencia del preámbulo
def plotSpect(signal, samplingRate):
    plt.subplots(figsize=(15, 3))
    plt.grid()
    plt.title("Power Spectral Density")
    plt.psd(signal, NFFT=len(signal), Fs=samplingRate, color="C1")
    plt.show()


plotSpect(preambulo_ref, SamplingRate)

In [ ]:
# TRANSMISOR — 6. Gráfico de la señal completa (trama)
trama = np.concatenate([preambulo_ref, modular(simbolos_payload)])
plotSignal(trama)

In [ ]:
# TRANSMISOR — 7. Densidad espectral de potencia de la trama completa
plotSpect(trama, SamplingRate)

In [ ]:
# TRANSMISOR — 8. Demodulación (Ecuaciones 15 y 16)
def demodular(recibido, M=M):
    """Demodula uno o varios símbolos por dechirp + FFT."""
    datos = np.asarray(recibido, dtype=np.complex128)
    if datos.ndim == 1:
        if datos.size % M != 0:
            raise ValueError("la señal no es múltiplo de M")
        datos = datos.reshape(-1, M)
    dechirpeado = datos * down_chirp_demod(M)[None, :]
    espectros = np.abs(np.fft.fft(dechirpeado, axis=1))
    return np.argmax(espectros, axis=1)


simbolos_recuperados = demodular(modular(simbolos_payload))
print("Demodulación ida y vuelta:", np.array_equal(simbolos_recuperados, simbolos_payload))

In [ ]:
# TRANSMISOR — 9. Armado de la trama: preámbulo + payload
def construir_trama(simbolos, M=M, n_up=n_up, n_down=n_down, cuarto=cuarto_down):
    """Preámbulo seguido del payload modulado."""
    return np.concatenate([preambulo(M, n_up, n_down, cuarto), modular(simbolos, M)])


def longitud_trama(M=M, n_simbolos=n_simbolos, n_up=n_up, n_down=n_down,
                   cuarto=cuarto_down):
    return longitud_preambulo(M, n_up, n_down, cuarto) + n_simbolos * M


trama = construir_trama(simbolos_payload)
print(f"Preámbulo: {preambulo_ref.size} muestras")
print(f"Payload:   {n_simbolos*M} muestras")
print(f"Trama:     {trama.size} muestras ({trama.size/SamplingRate*1e3:.3f} ms)")

In [ ]:
# TRANSMISOR — 10. Detección del preámbulo: correlación en tiempo y en frecuencia
def correlacion_tiempo(recibido, patron=preambulo_ref):
    """Correlación directa en el dominio del tiempo."""
    recibido = np.asarray(recibido, dtype=np.complex128)
    patron = np.asarray(patron, dtype=np.complex128)
    if recibido.size < patron.size:
        raise ValueError("la señal es más corta que el patrón")
    return np.abs(np.correlate(recibido, patron, mode="valid"))


def correlacion_fft(recibido, patron=preambulo_ref, circular=False):
    """|correlación| calculada por FFT. Acepta una captura o un lote 2D."""
    recibido = np.asarray(recibido)
    patron = np.asarray(patron)
    if recibido.shape[-1] < patron.size:
        raise ValueError("la señal es más corta que el patrón")
    if circular:
        producto = np.fft.fft(recibido, axis=-1) * np.conj(np.fft.fft(patron))
        return np.abs(np.fft.ifft(producto, axis=-1))
    n = recibido.shape[-1] + patron.size - 1
    nfft = 1 << (n - 1).bit_length()
    producto = np.fft.fft(recibido, nfft, axis=-1) * np.conj(np.fft.fft(patron, nfft))
    completo = np.fft.ifft(producto, axis=-1)
    return np.abs(completo[..., : recibido.shape[-1] - patron.size + 1])


def detectar_tiempo(recibido, patron=preambulo_ref):
    """Inicio y métrica del preámbulo por correlación temporal."""
    recibido = np.asarray(recibido, dtype=np.complex128)
    correlacion = correlacion_tiempo(recibido, patron)
    inicio = int(np.argmax(correlacion))
    energia = np.sum(np.abs(recibido[inicio: inicio + patron.size]) ** 2)
    metrica = correlacion[inicio] / (np.linalg.norm(patron) * np.sqrt(energia))
    return {"inicio": inicio, "metrica": float(metrica), "correlacion": correlacion}


def detectar_frecuencia(recibido, patron=preambulo_ref, circular=False):
    """Inicio y métrica del preámbulo por correlación en frecuencia (FFT)."""
    recibido = np.asarray(recibido)
    unico = recibido.ndim == 1
    lote = recibido.reshape(1, -1) if unico else recibido
    correlacion = correlacion_fft(lote, patron, circular)
    inicio = np.argmax(correlacion, axis=-1)
    if circular:
        energia = np.sum(np.abs(lote) ** 2, axis=-1)
    else:
        potencia = np.abs(lote) ** 2
        acumulada = np.cumsum(
            np.concatenate([np.zeros((lote.shape[0], 1), dtype=potencia.dtype), potencia], axis=1),
            axis=1)
        filas = np.arange(lote.shape[0])
        energia = acumulada[filas, inicio + patron.size] - acumulada[filas, inicio]
    metrica = correlacion[np.arange(lote.shape[0]), inicio] / (
        np.linalg.norm(patron) * np.sqrt(energia))
    if unico:
        return {"inicio": int(inicio[0]), "metrica": float(metrica[0]),
                "correlacion": correlacion[0]}
    return {"inicio": inicio, "metrica": metrica, "correlacion": correlacion}


# Demostración: la trama desplazada y con ruido debe dar el mismo inicio por los dos métodos.
margen = 64
captura_demo = np.zeros(trama.size + 2 * margen, dtype=np.complex128)
captura_demo[margen: margen + trama.size] = trama
generador_demo = np.random.default_rng(semilla + 1)
ruido_demo = (generador_demo.normal(size=captura_demo.size)
              + 1j * generador_demo.normal(size=captura_demo.size))
potencia_demo = np.mean(np.abs(trama) ** 2)
captura_demo += ruido_demo * np.sqrt(potencia_demo / 2)

info_tiempo = detectar_tiempo(captura_demo)
info_frecuencia = detectar_frecuencia(captura_demo)
print(f"Inicio real: {margen}")
print(f"Tiempo     : {info_tiempo['inicio']} (métrica {info_tiempo['metrica']:.3f})")
print(f"Frecuencia : {info_frecuencia['inicio']} (métrica {info_frecuencia['metrica']:.3f})")

In [ ]:
# TRANSMISOR — 11. FER en simulación contra SNR (se evalúa solo el preámbulo)
def fer_simulada(selectivo, snr_db, n_tramas, lote=2500, semilla=semilla):
    """FER del preámbulo: falla si el inicio detectado no coincide con el real."""
    generador = np.random.default_rng(semilla)
    snr_muestra = snr_db - 10 * np.log10(M / SF)   # SNR referida a Eb/N0
    patron = np.asarray(preambulo_ref, dtype=np.complex64)
    potencia = np.mean(np.abs(patron) ** 2)
    fallas = 0
    hechas = 0
    while hechas < n_tramas:
        b = min(lote, n_tramas - hechas)
        desplazamientos = generador.integers(0, patron.size, size=b)
        capturas = np.empty((b, patron.size), dtype=np.complex64)
        for i, desplazamiento in enumerate(desplazamientos):
            capturas[i] = np.roll(patron, desplazamiento)
        if selectivo:
            ecos = np.array([1.0, 0.9 * np.exp(1j * 2.1), 0.7 * np.exp(1j * 5.3)],
                            dtype=np.complex64)
            ecos = ecos / np.sqrt(np.sum(np.abs(ecos) ** 2))
            con_ecos = np.zeros_like(capturas)
            for eco, retardo in zip(ecos, (0, 20, 50)):
                con_ecos += eco * np.roll(capturas, retardo, axis=1)
            capturas = con_ecos
        ruido = (generador.normal(size=capturas.shape)
                 + 1j * generador.normal(size=capturas.shape)).astype(np.complex64)
        capturas += ruido * np.sqrt(potencia / (10 ** (snr_muestra / 10)) / 2)
        inicio = detectar_frecuencia(capturas, patron, circular=True)["inicio"]
        fallas += int(np.count_nonzero(inicio != desplazamientos))
        hechas += b
    return fallas / n_tramas


tramas_por_punto = n_tramas_fer // snr_fer_db.size
fer_flat = np.array([fer_simulada(False, snr, tramas_por_punto, semilla=semilla + i)
                     for i, snr in enumerate(snr_fer_db)])
fer_selectivo = np.array([fer_simulada(True, snr, tramas_por_punto, semilla=semilla + 100 + i)
                          for i, snr in enumerate(snr_fer_db)])

print(f"{'SNR [dB]':>9} {'FER FLAT':>10} {'FER selectivo':>14}")
for snr, plano, selectivo in zip(snr_fer_db, fer_flat, fer_selectivo):
    print(f"{snr:>9.0f} {plano:>10.2f} {selectivo:>14.2f}")

piso = 1 / (2 * tramas_por_punto)
plt.figure(figsize=(7, 4.5))
plt.semilogy(snr_fer_db, np.maximum(fer_flat, piso), "o-", label="FLAT (solo ruido)")
plt.semilogy(snr_fer_db, np.maximum(fer_selectivo, piso), "s-", label="selectivo (3 ecos)")
plt.xlabel("SNR = Eb/N0 [dB]")
plt.ylabel("FER")
plt.title("FER del preámbulo en simulación")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# CONFIGURACIÓN DEL SDR — se deja el enlace abierto para el receptor
import adi

sdr = adi.Pluto("ip:192.168.1.34")

# Si otra sesión dejó el chip en "alert", el RX no entrega muestras.
if sdr._ctrl.attrs["ensm_mode"].value != "fdd":
    sdr._ctrl.attrs["ensm_mode"].value = "fdd"

sdr.loopback = 0
sdr.sample_rate = int(SamplingRate)
sdr.tx_rf_bandwidth = 2_000_000
sdr.rx_rf_bandwidth = 2_000_000
sdr.tx_hardwaregain_chan0 = -60
sdr.gain_control_mode_chan0 = "slow_attack"
sdr.tx_lo = int(fc)
sdr.rx_lo = int(fc)
sdr.rx_buffer_size = tramas_en_buffer * longitud_trama()
sdr.tx_cyclic_buffer = True

print(f"Enlace configurado: {fc/1e6:.0f} MHz | RX buffer {sdr.rx_buffer_size} muestras")
print("El buffer cíclico se activa antes de transmitir.")

In [ ]:
# ENVÍO DE LA SEÑAL AL SDR
sdr.tx(trama * (2 ** 15 - 1))
print("Trama transmitida; el buffer cíclico la repite sin pausa.")

In [ ]:
# RECEPTOR — 1. Gráfico de la señal recibida
captura = sdr.rx()
plotSignal(captura)

In [ ]:
# RECEPTOR — 2. Densidad espectral de potencia de la señal recibida
plotSpect(captura, SamplingRate)

In [ ]:
# RECEPTOR — 3. FER con 20000 tramas (sin gráfico)
n_tramas_enlace = 20000
umbral_deteccion = 0.5
tramas_ok = 0
tramas_mal = 0

for indice in range(n_tramas_enlace):
    captura = sdr.rx()
    info = detectar_frecuencia(captura)
    if info["metrica"] >= umbral_deteccion:
        tramas_ok += 1
    else:
        tramas_mal += 1
    if (indice + 1) % 2000 == 0:
        print(f"{indice + 1}/{n_tramas_enlace} tramas | errores: {tramas_mal}")

fer_enlace = tramas_mal / n_tramas_enlace
print(f"FER con {n_tramas_enlace} tramas: {fer_enlace:.6f}")
print(f"  detectadas: {tramas_ok} | falladas: {tramas_mal}")

In [ ]:
# CIERRE DEL ENLACE — dejar el Pluto atenuado y fuera de la banda
sdr.tx_destroy_buffer()
sdr.tx_hardwaregain_chan0 = -89
sdr.tx_lo = 2400e6
sdr.rx_lo = 950e6
sdr.tx(np.zeros(2048, dtype=np.complex128))
print("Enlace cerrado: TX atenuado y portadoras fuera de la banda.")